# Part 1 · Cache the text embeddings and image latents

**Why this part exists:** the big FLUX memory costs are T5 (~10 GB), the VAE and the 12B transformer. Your captions and images are fixed, so T5/CLIP/VAE only need to run **once**. This part runs them, saves the results, and from then on training and generation never load them again.

- **Needs:** any GPU with ~12 GB (T5 runs in bf16; a T4 is fine, it is only 200 captions) · a ~10 GB download · roughly 10-15 min
- **Needs from before:** `pilot_v1.zip` (notebook 01)
- **Produces:** `cache/cache_r256.pt` (a few hundred MB) in your store
- **Settings:** `RES=256` is the photos' native size and keeps training cheap. `MAX_SEQ=128` T5 tokens (your captions are short; the cell prints how many would be cut). Change them only if you also re-run everything after this part.

In [ ]:
# --- bootstrap (identical in every part): find the pipeline files, set paths ---
import os, glob, shutil, sys
ON_KAGGLE = os.path.exists('/kaggle/working')
if ON_KAGGLE:
    hits = glob.glob('/kaggle/input/**/common.py', recursive=True)
    assert hits, 'Upload the pipeline/*.py files + pilot_v1.zip as a Kaggle dataset and add it as an input to this notebook.'
    SRC, WORK = os.path.dirname(hits[0]), '/kaggle/working'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    SRC, WORK = '/content/drive/MyDrive/sketchy_dataset_one_sketch/pipeline', '/content'
os.makedirs(f'{WORK}/pipeline', exist_ok=True)
for f in glob.glob(f'{SRC}/*.py'):
    shutil.copy(f, f'{WORK}/pipeline/')
sys.path.insert(0, f'{WORK}/pipeline')
from common import *
print('platform:', 'Kaggle' if ON_KAGGLE else 'Colab', '| STORE =', STORE)

Mounted at /content/drive


In [ ]:
!pip install -q diffusers==0.33.1 transformers==4.52.4 accelerate==1.7.0 sentencepiece protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 128.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.1/362.1 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 111.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import torch
p = torch.cuda.get_device_properties(0)
print(p.name, round(p.total_memory / 1e9), 'GB | bf16:', torch.cuda.get_device_capability()[0] >= 8)

Tesla T4 16 GB | bf16: False


In [ ]:
from huggingface_hub import login
login()   # FLUX.1-schnell is gated: accept the terms on its Hugging Face page first, then use a read token

In [ ]:
from huggingface_hub import whoami, HfApi
from google.colab import userdata
import os

tok = None
try:
    tok = userdata.get('HF_TOKEN')
except Exception as e:
    print('Could not find HF_TOKEN in Colab Secrets:', type(e).__name__)
    # Fallback to environment variable or manual entry if needed
    tok = os.environ.get('HF_TOKEN')

if not tok:
    print('Please add your HF_TOKEN in the Colab Secrets tab on the left panel.')
else:
    print('token length:', len(tok))
    try:
        print('logged in as:', whoami(token=tok)['name'])
        HfApi().model_info('black-forest-labs/FLUX.1-schnell', token=tok)
        print('access: OK')
    except Exception as e:
        print('access: NO ->', type(e).__name__)

token length: 37
logged in as: harshi282828
access: OK


In [ ]:
import sys, os, glob, shutil

# Automatically bootstrap if /content/pipeline is missing
pipeline_path = '/content/pipeline'
if not os.path.exists(pipeline_path) or not os.listdir(pipeline_path):
    print("Pipeline directory missing. Re-mounting drive and copying files...")
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    SRC = '/content/drive/MyDrive/sketchy_dataset_one_sketch/pipeline'
    os.makedirs(pipeline_path, exist_ok=True)
    for f in glob.glob(f'{SRC}/*.py'):
        shutil.copy(f, f'{pipeline_path}/')

if pipeline_path not in sys.path:
    sys.path.insert(0, pipeline_path)

from common import download, FLUX, TEXT_PARTS
download('black-forest-labs/FLUX.1-schnell', FLUX, allow=TEXT_PARTS)
print(os.listdir(FLUX))

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

model_index.json:   0%|          | 0.00/536 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/774 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

['scheduler', 'vae', 'text_encoder_2', 'text_encoder', 'model_index.json', 'tokenizer', 'tokenizer_2', '.cache']


In [ ]:
PILOT = f'{WORK}/pilot_v1'
if not os.path.exists(f'{PILOT}/metadata.jsonl'):
    shutil.unpack_archive(need('pilot_v1.zip'), PILOT)
print(open(f'{PILOT}/manifest.json').read()[:400])

{
  "version": "pilot_v1",
  "frozen_at": "2026-10-21T07:32:35",
  "resolution": 512,
  "seed": 0,
  "split_unit": "photo",
  "counts": {
    "source_rows": 200,
    "kept": 200,
    "train": 160,
    "val": 20,
    "test": 20
  },
  "dropped_by_reason": {},
  "approved_fallback": [],
  "approved_failed": [],
  "manual_drop": [],
  "metadata_sha256": "65113436087c1f773e543a4e798c825e64e9a6888a6051


In [ ]:
import importlib.util
import torch
import diffusers

print("PyTorch:", torch.__version__)
print("Diffusers:", diffusers.__version__)
print("TorchAO installed:", importlib.util.find_spec("torchao") is not None)

PyTorch: 2.11.0+cu130
Diffusers: 0.33.1
TorchAO installed: True


In [ ]:
import logging
import os

logging.basicConfig(level=logging.INFO)

print("Skipping optional TorchAO cache build.")
import logging
logging.basicConfig(level=logging.INFO)

Skipping optional TorchAO cache build.


In [ ]:
import importlib.metadata as md
print(md.version("torchao"))

0.18.0


In [ ]:
import sys
import logging
import builtins

# Inject a global fallback 'logger' into Python builtins to stop diffusers' bug from breaking imports
builtins.logger = logging.getLogger("diffusers")

try:
    from torchao.quantization import FqnToConfig
    # If this import succeeds, we can proceed to run the build
    import cache_build
    import os
    RES, MAX_SEQ = 256, 128
    os.makedirs(f'{STORE}/cache', exist_ok=True)
    cache = cache_build.bui-ld(PILOT, FLUX, f'{STORE}/cache/cache_r{RES}.pt', res=RES, max_seq=MAX_SEQ)
except ImportError:
    print("Upgrading torchao and restarting the session to load the updated version...")
    !pip install -q -U torchao
    import os
    os.kill(os.getpid(), 9)

Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

201 unique captions, 0 longer than 128 T5 tokens (those get truncated)
saved /content/drive/MyDrive/sketchy_dataset_one_sketch/cache/cache_r256.pt (231 MB), latent shape (16, 32, 32)


In [ ]:
import collections
print(collections.Counter(cache['split'].values()))
k = cache['ids'][0]
print({n: tuple(v.shape) for n, v in cache['lat'][k].items()}, '| text', tuple(cache['text'][cache['caption'][k]][0].shape))

Counter({'train': 160, 'val': 20, 'test': 20})
{'target': (16, 32, 32), 'sketch': (16, 32, 32), 'reference': (16, 32, 32)} | text (128, 4096)


**Kaggle only:** use *Save Version → Save & Run All* so `store/cache/...` becomes this notebook's output; add that output as an input to Parts 2-4.